Player 1 -> Minimax -> Defensive

Player 2 -> Expectimax -> Offensive

Player 3 -> USER/AI -> manual/ai

In [105]:
#card class
class Card:
  def __init__(self, color, number):
    self.card_color=color
    self.card_number=number
    self.card=(color, number)
  def __eq__(self, other):
    if other is None:
      return False
    return (self.card_color == other.card_color or
            self.card_number == other.card_number)
  #for nicer prints
  def __str__(self):
        return f"{self.card_color} {self.card_number}"
  def __repr__(self):
        return f"{self.card_color} {self.card_number}"
  def exact_match(self, other):
        return self.card_color == other.card_color and self.card_number == other.card_number
print("Card class created!")

Card class created!


In [106]:
#deck generator
import random #to get random number and color
colors=["Red", "Blue", "Green", "Yellow"]
values=["0", "1", "2", "3", "4", "5", "6", "7", "8", "9", "Skip"]
def shuffle_deck(deck):
  for color in colors:
    for value in values:
      deck.append(Card(color, value))
  random.shuffle(deck)
  return deck
deck=[]
deck=shuffle_deck(deck)
print("Deck Generated!")

Deck Generated!


In [107]:
#user choice
user_choice=0
while(True):
  user_choice=int(input("Press 1 for Manual and 2 for AI: "))
  if(user_choice==1 or user_choice==2):
    print("Let the game begin!")
    break
  else:
    print("Invalid choice.")

Press 1 for Manual and 2 for AI: 2
Let the game begin!


In [108]:
#Valid move check
def get_valid_moves(hand, top_card):
  #legal move generator
  valid_moves=[]
  #we go through the deck and throw in anything with the same color or number into valid moves
  for card in hand:
    if card==top_card:
      valid_moves.append(card)
  return valid_moves

In [109]:
#evaluation function
#different for offensive and defensive
def evaluaiton_function_offenive(num_cards, avg_opponent, skip_in_hand):
  score=50-10*(num_cards)+1*(avg_opponent)+1*(skip_in_hand)
  return score
def evaluation_function_defensive(num_cards, avg_opponent, skip_in_hand):
  score=50-5*(num_cards)+5*(avg_opponent)+4*(skip_in_hand)
  return score
def evaluation_function(num_cards, avg_opponent, skip_in_hand):
  score=50-5*(num_cards)+2*(avg_opponent)+3*(skip_in_hand)
  return score

In [110]:
def count_skips(hand):
  count=0
  for card in hand:
    if card.card_number=="Skip":
      count+=1
  return count

In [111]:
#state transition
def apply_move(state, move, current_player):
#updating the state idk why we need to keep a state when i can just keep variables </3
#move is the new caard
#state is the old past
#currrent player is jsut to show whos plaaayer rn
    new_state = {
        "Player1_hand": state["Player1_hand"].copy(),
        "Player2_hand": state["Player2_hand"].copy(),
        "Player3_hand": state["Player3_hand"].copy(),
        "Deck": state["Deck"].copy(),
        "Top_card": state["Top_card"],  # will update below
    }
    if current_player == 3:
        new_state["Player3_hand"].remove(move)
    elif current_player == 1:
        new_state["Player1_hand"].remove(move)
    else:
        new_state["Player2_hand"].remove(move)
    new_state["Top_card"] = move
    return new_state

#movement code
def pick_best_move_for_player1(player1_hand, player2_hand, player3_hand, deck, depth, valid_moves, top_card):
  state = {
      "Player1_hand": player1_hand.copy(),
      "Player2_hand": player2_hand.copy(),
      "Player3_hand": player3_hand.copy(),
      "Deck": deck.copy(),
      "Top_card": top_card
  }
  best_score = float('inf')
  best_move = "Dookie"
  for move in valid_moves:
    #current player 1
    #we applyy the current move and see how well they do and get their scores
    #and since this is min we do min value
    new_state = apply_move(state, move, 1)
    score = minmax_search(
            new_state,
            depth - 1,
            2,
            []
        )
    if score < best_score:
      best_score = score
      best_move = move
  return best_move

def pick_best_move_for_player3(player1_hand, player2_hand, player3_hand, deck, depth, valid_moves, user_choice, top_card):
  #here we wwillcheck if its the user playing or the AI playinh
  #need to fix this part where player selects a card
  if user_choice == 1:
      print("Pick a card to play: ")
      for card in valid_moves:
        print(card)
      card_obj=None
      while(True):
        color=input("Enter color: ")
        number=input("Enter number: ")
        card_obj=Card(color, number)
        if not any(card_obj.exact_match(c) for c in valid_moves):
          print("Invalid card.")
        else:
          break
      return card_obj
  else:
      #ai mode
      state = {
        "Player1_hand": player1_hand.copy(),
        "Player2_hand": player2_hand.copy(),
        "Player3_hand": player3_hand.copy(),
        "Deck": deck.copy(),
        "Top_card": top_card
      }
      best_score = float('-inf')
      best_move = "Dookie"
      for move in valid_moves:
        #current player 3
        #we applyy the current move and see how well they do and get their scores
        #and since this is max we do max value
        new_state = apply_move(state, move, 3)
        score = minmax_search(
              new_state,
              depth - 1,
              1, #next  player
              []
          )
        if score > best_score:
          best_score = score
          best_move = move
      return best_move

def pick_best_move_for_player2(player1_hand, player2_hand, player3_hand, deck, depth, valid_moves, top_card):
  state = {
      "Player1_hand": player1_hand.copy(),
      "Player2_hand": player2_hand.copy(),
      "Player3_hand": player3_hand.copy(),
      "Deck": deck.copy(),
      "Top_card": top_card
  }
  best_score=float('inf')
  best_move=None
  for move in valid_moves:
    new_state=apply_move(state, move, 2)
    score=expectimax_search(new_state, depth-1, 3, [], 0)
    if best_move is None or score < best_score:
      best_score = score
      best_move = move
  return best_move

In [112]:
#expectimax search
#usedBY P2 P2 IS OFFESIVE
def expectimax_search(state, depth, current_player, valid_moves, indent=0):
  #P1-RANDOM
  #P2->CHANCE
  #P3->MAX
  if current_player==1:
    hand=state["Player1_hand"]
  elif current_player==2:
    hand=state["Player2_hand"]
  else:
    hand=state["Player3_hand"]
  top_card=state["Top_card"]
  valid_moves=get_valid_moves(hand, top_card)
  prefix="|  "*indent
  #base case
  #SCORING IS ALWAYS FROM THE PERSPECTIVE OF MAX
  if depth==0:
    skips=count_skips(state["Player3_hand"])
    avg=(len(state["Player2_hand"])+len(state["Player1_hand"]))/2
    score=evaluation_function(len(state["Player3_hand"]), avg , skips)
    print(f"{prefix}└── [Player {current_player}] LEAF → score: {score:.2f}")
    return score
  else:
    #agaim player 1 and 2 are opponents so they are min
    #and plsyer 3 is us so we are maxxing AURAMAXXING
    if current_player==3:
      max_value=-9999
      for move in valid_moves:
        print(f"{prefix}├── [Player {current_player}] plays: {move}")
        new_state=apply_move(state, move, current_player)
        value=expectimax_search(new_state, depth-1, 1, [],  indent+1)
        max_value=max(max_value, value)
      return max_value
    elif current_player==2:
      valid_moves=get_valid_moves(state["Player2_hand"], top_card)
      if valid_moves:
        best=-9999
        for move in valid_moves:
          new_state=apply_move(state, move, 2)
          value=expectimax_search(new_state, depth-1, 3, [], indent+1)
          best=max(best, value)
        return best
      #treat draw card as chance node
      deck=state["Deck"]
      if not deck:
        #wwhat to do if deck is empty
        skips = count_skips(state["Player3_hand"])
        avg = (len(state["Player2_hand"]) + len(state["Player1_hand"])) / 2
        return evaluation_function(len(state["Player3_hand"]), avg, skips)
      else:
        #deck isnt empty
        total_value=0
        for card in deck:
          prob=1/len(deck)#each card in deck is equally likely  to  be draawn
          #making a copy of the staate
          new_state={k: v[:] if isinstance(v, list) else v for k, v in state.items()}
          new_state["Player2_hand"]=state["Player2_hand"]+[card]
          value=expectimax_search(new_state, depth-1, 3, [],  indent+1)
          total_value+=prob*value
        return total_value
    else:
      #p3 here is supposed to be random
      if not valid_moves:
        skips = count_skips(state["Player3_hand"])
        avg = (len(state["Player2_hand"]) + len(state["Player1_hand"])) / 2
        return evaluation_function(len(state["Player3_hand"]), avg , skips)
      move=random.choice(valid_moves)
      print(f"{prefix}├── [P1 RANDOM] plays: {move}")
      new_state=apply_move(state, move, 1)
      return expectimax_search(new_state, depth-1, 2, [], indent+1)







In [113]:
#minmax search
#USED BY P1 and AI P3
#THEY ARE MORE DEFENSIVE
def minmax_search(state, depth, current_player, valid_moves, indent=0):
  #i genuinely do not understand how to code this
  #my lord hep me pls
  #P3-> MAX
  #P1->MIN
  #P2->MIN
  if current_player==1:
    hand=state["Player1_hand"]
  elif current_player==2:
    hand=state["Player2_hand"]
  else:
    hand=state["Player3_hand"]
  top_card=state["Top_card"]
  valid_moves=get_valid_moves(hand, top_card)
  prefix="│  "*indent
  #base case
  if depth==0:
    skips=count_skips(state["Player3_hand"])
    avg=(len(state["Player2_hand"])+len(state["Player1_hand"]))/2
    score=evaluation_function_defensive(len(state["Player3_hand"]), avg , skips)
    print(f"{prefix}└── [P{current_player}] LEAF → score: {score:.2f}")
    return score
  else:
    if current_player==3:
      #this is the max node
      max_value=-9999
      #now i need to call recursivelly and get values from beloww nodes
      for move in valid_moves:
        print(f"{prefix}├── [P3 MAX] plays: {move}")
        new_state=apply_move(state, move, current_player)
        value=minmax_search(
        new_state,
        depth-1,
        1, #next player
        [],
        indent+1
        )
        max_value=max(max_value, value)
      return max_value
    #APPLYING MIN FOR BOTH P2 AND P1
    elif current_player==2:
      min_value=9999
      #need to cll recursively
      for move in valid_moves:
        print(f"{prefix}├── [P2 MIN] plays: {move}")
        new_state=apply_move(state, move, current_player)
        value=minmax_search(
        new_state,
        depth-1,
        3,
        [],
        indent+1
        )
        min_value=min(min_value, value)
      return min_value
    else:
      #also min node
      min_value=9999
      #need to cll recursively
      for move in valid_moves:
        print(f"{prefix}├── [P1 MIN] plays: {move}")
        new_state=apply_move(state, move, current_player)
        value=minmax_search(
        new_state,
        depth-1,
        2,
        [],
        indent+1
        )
        min_value=min(min_value, value)
      return min_value
  #max is always player 3 becuase thats us and we are MAX
  #player 2 and 3 are MIN they will try to minimize our score


  pass

In [114]:
#hand empty?
def deck_empty(deck):
  if not deck:
    return True
  else:
    return False

In [115]:
#game simulation
def play_game(player1_hand, player2_hand, player3_hand, deck, user_choice, result=""):
  RESET='\u001b[0m'
  RED='\u001b[31m'
  GREEN='\u001b[32m'
  BLUE='\u001b[34m'
  BOLD_CYAN='\u001b[1;36m'
  cards_played=[]
  skip_next_player=None
  players_won=[]

  while(True):

    if len(players_won) == 2:
      print("2 players won.")
      result+="2 players won.\n"
      break

    if not deck and not get_valid_moves(player1_hand, cards_played[-1]) and not get_valid_moves(player2_hand, cards_played[-1]) and not get_valid_moves(player3_hand, cards_played[-1]):
        print("Deck empty and no valid moves. Game over.")
        result+="Deck empty and no valid moves. Game over.\n"
        break

    #player 1
    if deck_empty(player1_hand):
      if "Player1" not in players_won:
        players_won.append("Player1")
    elif skip_next_player==1:
      print("Skipping player 1's turn")
      result+="Skipping player 1's turn\n"
      skip_next_player=None
    else:
      if not cards_played:
        print("No cards placed down right now.")
        result+="No cards placed down right now.\n"
        valid_moves=player1_hand
        card_played=pick_best_move_for_player1(player1_hand, player2_hand, player3_hand, deck, 3, valid_moves, None)
        player1_hand.remove(card_played)
        cards_played.append(card_played)
        print(f"{RED}   Player 1 played: {card_played}  {RESET}")
        result+=f"{RED}   Player 1 played: {card_played}  {RESET}\n"
        if card_played.card_number=="Skip":
          skip_next_player=2
      else:
        top_card=cards_played[-1] #getting last card placed
        valid_moves=get_valid_moves(player1_hand, top_card)
        if not valid_moves:
          if deck:
            card_drawn=deck[0]
            print(f"{RED}   Player 1 drawed: {card_drawn}  {RESET}")
            result+=f"{RED}   Player 1 drawed: {card_drawn}  {RESET}\n"
            player1_hand.append(deck[0])
            deck.pop(0)
        else:
          card_played=pick_best_move_for_player1(player1_hand, player2_hand, player3_hand, deck, 3, valid_moves, top_card)
          player1_hand.remove(card_played)
          cards_played.append(card_played)
          print(f"{RED}   Player 1 played: {card_played}  {RESET}")
          result+=f"{RED}   Player 1 played: {card_played}  {RESET}\n"
          if card_played.card_number=="Skip":
            skip_next_player=2

    #winning condiiton check
    if len(players_won) == 2:
      print("2 players won.")
      result+="2 players won.\n"
      break

    #player 2
    if deck_empty(player2_hand):
      if "Player2" not in players_won:
        players_won.append("Player2")
    elif skip_next_player==2:
      print("Skipping player 2's turn")
      result+="Skipping player 2's turn\n"
      skip_next_player=None
    else:
      top_card=cards_played[-1]
      valid_moves=get_valid_moves(player2_hand, top_card)
      if not valid_moves:
          if deck:
            card_drawn=deck[0]
            print(f"{BLUE}   Player 2 drawed: {card_drawn}  {RESET}")
            result+=f"{BLUE}   Player 2 drawed: {card_drawn}  {RESET}\n"
            player2_hand.append(deck[0])
            deck.pop(0)
      else:
          card_played=pick_best_move_for_player2(player1_hand, player2_hand, player3_hand, deck, 3, valid_moves, top_card)
          player2_hand.remove(card_played)
          cards_played.append(card_played)
          print(f"{BLUE}   Player 2 played: {card_played}  {RESET}")
          result+=f"{BLUE}   Player 2 played: {card_played}  {RESET}\n"
          if card_played.card_number=="Skip":
            skip_next_player=3

    #winning condiitton check
    if len(players_won) == 2:
      print("2 players won.")
      result+="2 players won.\n"
      break

    #player 3
    if deck_empty(player3_hand):
      print("Player 3 has won, skipping turn.")
      result+="Player 3 has won, skipping turn.\n"
      if "Player3" not in players_won:
        players_won.append("Player3")
    elif skip_next_player==3:
      print("Skipping player 3's turn")
      result+="Skipping player 3's turn\n"
      skip_next_player=None
    else:
      top_card=cards_played[-1]
      valid_moves=get_valid_moves(player3_hand, top_card)
      if not valid_moves:
        if deck:
          card_drawn=deck[0]
          print(f"{GREEN}   Player 3 drawed: {card_drawn}  {RESET}")
          result+=f"{GREEN}   Player 3 drawed: {card_drawn}  {RESET}\n"
          player3_hand.append(deck[0])
          deck.pop(0)
      else:
          card_played=pick_best_move_for_player3(player1_hand, player2_hand, player3_hand, deck, 3, valid_moves, user_choice, top_card)
          player3_hand.remove(card_played)
          cards_played.append(card_played)
          print(f"{GREEN}   Player 3 played: {card_played}  {RESET}")
          result+=f"{GREEN}   Player 3 played: {card_played}  {RESET}\n"
          if card_played.card_number=="Skip":
            skip_next_player=1
    #results
    if len(players_won) == 2:
      print("2 players won.")
      result+="2 players won.\n"
      break
  print("Players won: ", players_won)
  result+=f"Players won: {players_won}\n"
  return result

In [116]:
#giving starting cards to each player
def distribute_cards(player1_deck, player2_deck, player3_deck, deck):
  #each player gets 7 cards
  for i in range(7):
    player1_deck.append(deck[0])
    deck.pop(0)
    player2_deck.append(deck[0])
    deck.pop(0)
    player3_deck.append(deck[0])
    deck.pop(0)
  return player1_deck, player2_deck, player3_deck, deck

RESET='\u001b[0m'
RED='\u001b[31m'
GREEN='\u001b[32m'
BLUE='\u001b[34m'
BOLD_CYAN='\u001b[1;36m'
player1_hand=[]
player2_hand=[]
player3_hand=[]
player1_hand, player2_hand, player3_hand, deck=distribute_cards(player1_hand, player2_hand, player3_hand, deck)
print("Cards distributed!")
print("Starting game: ")
print(play_game(player1_hand, player2_hand, player3_hand, deck, user_choice))


Cards distributed!
Starting game: 
No cards placed down right now.
├── [P2 MIN] plays: Green 8
├── [P2 MIN] plays: Blue 0
│  ├── [P3 MAX] plays: Blue 1
│  │  └── [P1] LEAF → score: 50.00
│  ├── [P3 MAX] plays: Blue 2
│  │  └── [P1] LEAF → score: 50.00
│  ├── [P3 MAX] plays: Blue 3
│  │  └── [P1] LEAF → score: 50.00
│  ├── [P3 MAX] plays: Blue 9
│  │  └── [P1] LEAF → score: 50.00
├── [P2 MIN] plays: Red 6
│  ├── [P3 MAX] plays: Red 2
│  │  └── [P1] LEAF → score: 50.00
├── [P2 MIN] plays: Red 3
│  ├── [P3 MAX] plays: Blue 3
│  │  └── [P1] LEAF → score: 50.00
│  ├── [P3 MAX] plays: Red 2
│  │  └── [P1] LEAF → score: 50.00
│  ├── [P3 MAX] plays: Yellow 3
│  │  └── [P1] LEAF → score: 50.00
├── [P2 MIN] plays: Blue 0
│  ├── [P3 MAX] plays: Blue 1
│  │  └── [P1] LEAF → score: 50.00
│  ├── [P3 MAX] plays: Blue 2
│  │  └── [P1] LEAF → score: 50.00
│  ├── [P3 MAX] plays: Blue 3
│  │  └── [P1] LEAF → score: 50.00
│  ├── [P3 MAX] plays: Blue 9
│  │  └── [P1] LEAF → score: 50.00
├── [P2 MIN] plays: